# 03 · Feature Engineering

**Step 3** — build the six normalised **operational flood indicators** and generate physically-motivated flood labels.

| Indicator | Meaning |
| --- | --- |
| `low_elevation_susceptibility` | lower ground floods first |
| `rainfall_intensity` | normalised rainfall load |
| `drainage_stress` | wetness + drainage-density load |
| `slope_vulnerability` | flat ground drains poorly |
| `runoff_potential` | upslope contributing area |
| `terrain_instability` | ruggedness + concavity |

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, matplotlib.pyplot as plt
from src import feature_engineering, terrain_analysis, hydrology, utils
from src.feature_engineering import INDICATOR_NAMES
from src.utils import STUDY_AREA, PROCESSED_DIR

dem = np.load(PROCESSED_DIR / 'dem.npy')
rainfall_field = np.load(PROCESSED_DIR / 'rainfall_field.npy')
terrain = terrain_analysis.analyze_terrain(dem)
hydro = hydrology.analyze_hydrology(dem, terrain['slope'])
extent = [STUDY_AREA.min_lon, STUDY_AREA.max_lon, STUDY_AREA.min_lat, STUDY_AREA.max_lat]

In [ ]:
feats = feature_engineering.engineer_features(terrain, hydro, rainfall_field)
indicators = feats['indicators']
table = feats['table']
table.head()

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(16, 9))
for a, name in zip(ax.ravel(), INDICATOR_NAMES):
    im = a.imshow(indicators[name], cmap='RdYlGn_r', extent=extent, vmin=0, vmax=1)
    a.set_title(name.replace('_',' ').title(), fontsize=10)
    fig.colorbar(im, ax=a, shrink=0.75)
plt.tight_layout(); plt.show()

## Label distribution & feature correlation

In [ ]:
print('Positive (flood) rate: %.1f%%' % (100*table['flood_label'].mean()))
corr = table[INDICATOR_NAMES + ['flood_label']].corr()
fig, ax = plt.subplots(figsize=(7,6))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=8)
fig.colorbar(im, shrink=0.8); ax.set_title('Feature correlation'); plt.tight_layout(); plt.show()

In [ ]:
for name, arr in indicators.items():
    np.save(PROCESSED_DIR / f'indicator_{name}.npy', arr)
try:
    table.to_parquet(PROCESSED_DIR / 'feature_table.parquet')
except Exception:
    table.to_csv(PROCESSED_DIR / 'feature_table.csv', index=False)
print('Saved indicators + feature table. Next: 04_model_training.ipynb')